# INF201 – Deliverable 1: Værdata fra Ås

Gruppe: Sigurd Ruud og Shem 
GitHub-repo:

I denne notebooken henter vi værdata fra Meteorologisk institutt sitt Frost-API
for værstasjonen på Ås (SN17850) og analyserer temperatur, nedbør og hetebølger.

Koden for å hente data bygger på eksempelet fra
[frost.met.no](https://frost.met.no/python_example.html), men er skrevet om til en gjenbrukbar funksjon.

## Oppsett

Klient-ID-en til Frost er personlig og skal **ikke** ligge i Git-repoet.
Vi lagrer den derfor i en tekstfil i hjemmemappen (`~/.frost_client_id`), utenfor repoet,
og leser den inn her. Filen skal bare inneholde selve ID-en på én linje.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import requests
import yaml

# Les klient-ID fra en fil utenfor repoet
CLIENT_ID_FILE = Path.home() / ".frost_client_id"
client_id = CLIENT_ID_FILE.read_text().strip()

ENDPOINT = "https://frost.met.no/observations/v0.jsonld"
STATION_ID = "SN17850"   # Ås
STATION_NAME = "Ås"

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": True, "grid.alpha": 0.3})

### Funksjon for å hente data fra Frost

Vi trenger å hente flere typer data (middeltemperatur, nedbør, maksimumstemperatur)
for ulike år, så vi lager én funksjon vi kan gjenbruke.

Frost svarer med en liste der hvert element er én dag (`referenceTime`) med en liste
av observasjoner. Noen dager kan ha flere observasjoner av samme element
(f.eks. med ulik `timeOffset`). Vi beholder derfor bare den første observasjonen per dag,
slik at vi får nøyaktig én verdi per dato.

In [ ]:
def get_frost_data(element, referencetime, source=STATION_ID, client_id=client_id):
    """Hent observasjoner fra Frost og returner en DataFrame med én verdi per dag.

    Parametre
    ---------
    element : str
        Elementkode i Frost, f.eks. 'mean(air_temperature P1D)'.
    referencetime : str
        Tidsperiode, f.eks. '2025-01-01/2025-12-31'.
    source : str
        Stasjons-ID, f.eks. 'SN17850'.

    Returnerer
    ----------
    pandas.DataFrame med datoer som indeks og kolonnen 'value'.
    """
    parameters = {
        "sources": source,
        "elements": element,
        "referencetime": referencetime,
    }
    response = requests.get(ENDPOINT, parameters, auth=(client_id, ""))
    json = response.json()

    if response.status_code != 200:
        raise RuntimeError(
            f"Feil {response.status_code}: {json['error']['message']} "
            f"({json['error']['reason']})"
        )

    rows = []
    for day in json["data"]:
        for obs in day["observations"]:
            rows.append({"date": day["referenceTime"], "value": obs["value"]})

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.drop_duplicates(subset="date", keep="first").set_index("date")
    return df

### Funksjon for oppsummeringstabell

Vi skal lage oppsummeringstabeller i både oppgave 1 og 3, så vi lager en funksjon
som tar inn én eller flere tidsserier og returnerer gjennomsnitt, median, min og maks.

In [ ]:
def summary_table(series_dict):
    """Lag en tabell med mean, median, min og max for hver tidsserie.

    series_dict: dict der nøkkelen er kolonnenavn og verdien er en pandas Series.
    """
    table = pd.DataFrame({
        name: {
            "Gjennomsnitt": s.mean(),
            "Median": s.median(),
            "Minimum": s.min(),
            "Maksimum": s.max(),
        }
        for name, s in series_dict.items()
    })
    return table.round(1)

## Oppgave 1: Temperatur

Vi henter døgnmiddeltemperaturen for hele 2025.

In [ ]:
temp_2025 = get_frost_data("mean(air_temperature P1D)", "2025-01-01/2025-12-31")
print(f"Antall dager med data: {len(temp_2025)}")
temp_2025.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(temp_2025.index, temp_2025["value"], color="tab:red", linewidth=1.2)
ax.axhline(0, color="black", linewidth=0.8)

ax.set_title(f"Døgnmiddeltemperatur på {STATION_NAME} i 2025")
ax.set_xlabel("Dato")
ax.set_ylabel("Temperatur (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_xlim(temp_2025.index.min(), temp_2025.index.max())

plt.tight_layout()
plt.show()

In [ ]:
table_2025 = summary_table({"2025 (°C)": temp_2025["value"]})
print(f"Døgnmiddeltemperatur på {STATION_NAME}, 2025")
print(table_2025.to_string())

**Diskusjon:** _Skriv noen setninger om hva plottet og tabellen viser – f.eks. når det var
varmest og kaldest, og om det er noe som skiller seg ut._

## Oppgave 2: Nedbør

Vi henter døgnnedbør (`sum(precipitation_amount P1D)`) for samme periode.

**Valg av plottype:**
- **Nedbør** vises som **søyler**. Nedbør er en mengde som har samlet seg opp i løpet av et døgn,
  og mange dager har 0 mm. Søyler viser godt både hvor mye det regnet og hvilke dager det var tørt.
- **Temperatur** vises som en **linje**, fordi den endrer seg gradvis fra dag til dag.
- Siden de to har ulike enheter (°C og mm), bruker vi **to y-akser**: temperatur til venstre
  og nedbør til høyre, med samme farge på aksen som på dataene.

In [ ]:
precip_2025 = get_frost_data("sum(precipitation_amount P1D)", "2025-01-01/2025-12-31")
print(f"Antall dager med data: {len(precip_2025)}")
print(f"Total nedbør i 2025: {precip_2025['value'].sum():.0f} mm")

In [ ]:
fig, ax_temp = plt.subplots()
ax_precip = ax_temp.twinx()   # andre y-akse som deler x-aksen

ax_precip.bar(precip_2025.index, precip_2025["value"], width=1.0,
              color="tab:blue", alpha=0.5, label="Nedbør (mm)")
ax_temp.plot(temp_2025.index, temp_2025["value"],
             color="tab:red", linewidth=1.2, label="Middeltemperatur (°C)")

# Tegn temperaturlinjen foran søylene
ax_temp.set_zorder(ax_precip.get_zorder() + 1)
ax_temp.patch.set_visible(False)

ax_temp.set_title(f"Temperatur og nedbør på {STATION_NAME} i 2025")
ax_temp.set_xlabel("Dato")
ax_temp.set_ylabel("Temperatur (°C)", color="tab:red")
ax_precip.set_ylabel("Nedbør (mm)", color="tab:blue")
ax_temp.tick_params(axis="y", colors="tab:red")
ax_precip.tick_params(axis="y", colors="tab:blue")
ax_precip.grid(False)

ax_temp.xaxis.set_major_locator(mdates.MonthLocator())
ax_temp.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax_temp.set_xlim(temp_2025.index.min(), temp_2025.index.max())

# Felles forklaring for begge aksene
lines = ax_temp.get_legend_handles_labels()
bars = ax_precip.get_legend_handles_labels()
ax_temp.legend(lines[0] + bars[0], lines[1] + bars[1], loc="upper left")

plt.tight_layout()
plt.show()

**Diskusjon:** _Når kom det mest nedbør? Ser dere noen sammenheng mellom temperatur og nedbør?_

## Oppgave 3: Historisk temperatur – 1925 mot 2025

Vi henter døgnmiddeltemperaturen for 1925, 100 år tidligere, fra samme stasjon.

For å plotte to ulike år i samme figur flytter vi datoene til et felles år, slik at
1. januar 1925 og 1. januar 2025 havner på samme sted langs x-aksen.

In [ ]:
temp_1925 = get_frost_data("mean(air_temperature P1D)", "1925-01-01/1925-12-31")
print(f"Antall dager med data i 1925: {len(temp_1925)}")

In [ ]:
def to_common_year(df, year=2000):
    """Flytt alle datoer til samme år, slik at ulike år kan plottes oppå hverandre.

    Vi bruker 2000 som standard fordi det er et skuddår, så 29. februar alltid finnes.
    """
    df = df.copy()
    df.index = df.index.map(lambda d: d.replace(year=year))
    return df


fig, ax = plt.subplots()
for df, label, color in [(temp_1925, "1925", "tab:blue"), (temp_2025, "2025", "tab:red")]:
    common = to_common_year(df)
    ax.plot(common.index, common["value"], label=label, color=color, linewidth=1.2, alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Døgnmiddeltemperatur på {STATION_NAME}: 1925 og 2025")
ax.set_xlabel("Måned")
ax.set_ylabel("Temperatur (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_xlim(pd.Timestamp("2000-01-01"), pd.Timestamp("2000-12-31"))
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
table_compare = summary_table({
    "1925 (°C)": temp_1925["value"],
    "2025 (°C)": temp_2025["value"],
})
table_compare["Forskjell (°C)"] = (table_compare["2025 (°C)"] - table_compare["1925 (°C)"]).round(1)

print(f"Døgnmiddeltemperatur på {STATION_NAME}: 1925 mot 2025")
print(table_compare.to_string())

**Diskusjon:** _Var 2025 varmere eller kaldere enn 1925? Hvilke deler av året skiller seg mest?
Husk at to enkeltår ikke er nok til å si noe sikkert om klimaendringer – et enkelt år kan
være uvanlig varmt eller kaldt._

## Oppgave 4: Hetebølger

I Norge er en hetebølge definert som **minst fem dager på rad** der
**maksimumstemperaturen er 27 °C eller mer**. Vi bruker derfor
`max(air_temperature P1D)`, ikke middeltemperaturen.

Fremgangsmåte i funksjonen:
1. Hent maksimumstemperatur for hele året.
2. Lag en rad for hver dato i året. Dager uten data får ingen verdi og regnes ikke som varme,
   slik at et hull i dataene bryter en periode.
3. Gå gjennom dagene i rekkefølge og tell hvor mange varme dager det har vært på rad.
   Når rekken brytes, sjekker vi om den var lang nok til å være en hetebølge.
4. Skriv resultatet til en YAML-fil i samme format som eksempelfilen.

Funksjonen tar stasjons-ID og år som parametre, så den fungerer for alle stasjoner og år.
Hvis vi ikke oppgir stasjonsnavnet, hentes det automatisk fra Frost.

In [ ]:
def get_station_name(station_id, client_id=client_id):
    """Hent navnet på en værstasjon fra Frost."""
    response = requests.get(
        "https://frost.met.no/sources/v0.jsonld",
        {"ids": station_id},
        auth=(client_id, ""),
    )
    response.raise_for_status()
    return response.json()["data"][0]["name"]


def find_heatwaves(max_temp, threshold=27.0, min_days=5):
    """Finn hetebølger i en tidsserie med maksimumstemperatur per dag.

    Returnerer en liste med (startdato, sluttdato) for hver hetebølge.
    """
    heatwaves = []
    run_start = None   # første dag i en pågående rekke av varme dager
    run_length = 0
    previous_day = None

    for day, temp in max_temp.items():
        if pd.notna(temp) and temp >= threshold:
            if run_length == 0:
                run_start = day
            run_length += 1
        else:
            if run_length >= min_days:
                heatwaves.append((run_start, previous_day))
            run_length = 0
        previous_day = day

    # Hvis året slutter midt i en hetebølge
    if run_length >= min_days:
        heatwaves.append((run_start, previous_day))

    return heatwaves


def count_heatwaves(station_id, year, station_name=None, filename=None,
                    threshold=27.0, min_days=5):
    """Tell hetebølger for en stasjon og et år, og skriv en oppsummering til en YAML-fil.

    Parametre
    ---------
    station_id : str
        Stasjons-ID i Frost, f.eks. 'SN17850'.
    year : int
        Året som skal undersøkes.
    station_name : str, valgfri
        Stasjonsnavn. Hentes fra Frost hvis det ikke oppgis.
    filename : str, valgfri
        Navn på YAML-filen. Standard er 'heatwave_summary_<stasjon>_<år>.yml'.

    Returnerer
    ----------
    int : antall hetebølger
    """
    if station_name is None:
        station_name = get_station_name(station_id)
    if filename is None:
        filename = f"heatwave_summary_{station_id}_{year}.yml"

    max_temp = get_frost_data("max(air_temperature P1D)",
                              f"{year}-01-01/{year}-12-31", source=station_id)

    # Én rad per dag i året; dager uten data blir NaN
    all_days = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="D")
    max_temp = max_temp["value"].reindex(all_days)

    heatwaves = find_heatwaves(max_temp, threshold, min_days)

    summary = {
        "dates": [f"{start:%Y-%m-%d} - {end:%Y-%m-%d}" for start, end in heatwaves],
        "station_id": station_id,
        "station_name": station_name,
        "year": year,
    }
    with open(filename, "w") as f:
        yaml.dump(summary, f)

    return len(heatwaves)

In [ ]:
n = count_heatwaves(STATION_ID, 2025, station_name=STATION_NAME)
print(f"Antall hetebølger på {STATION_NAME} i 2025: {n}")
print()
print(open(f"heatwave_summary_{STATION_ID}_2025.yml").read())

**Diskusjon:** _Hvor mange hetebølger var det, og når? Dere kan også teste funksjonen på andre
år eller stasjoner, f.eks. Oslo – Blindern (`SN18700`), for å vise at den er generell._

## Oppgave 5: Glidende gjennomsnitt

Et 7-dagers glidende gjennomsnitt erstatter hver dag med gjennomsnittet av 7 dager rundt den.
Det glatter ut svingninger fra dag til dag, slik at den langsiktige trenden blir lettere å se.

Vi bruker et **sentrert** vindu: verdien for en dag er gjennomsnittet av dagen selv,
de 3 dagene før og de 3 dagene etter. Da havner kurven på riktig sted i tid i plottet.
De 3 første og 3 siste dagene i året får ingen verdi, fordi det ikke finnes nok dager rundt dem.

In [ ]:
def moving_average(series, window=7):
    """Beregn sentrert glidende gjennomsnitt over et vindu på `window` dager.

    Dager som ikke har et helt vindu rundt seg (starten og slutten av serien) blir NaN.
    """
    return series.rolling(window=window, center=True, min_periods=window).mean()


temp_2025_ma = moving_average(temp_2025["value"], window=7)

In [ ]:
fig, ax = plt.subplots()
ax.scatter(temp_2025.index, temp_2025["value"], s=10, color="tab:gray",
           alpha=0.6, label="Daglig middeltemperatur")
ax.plot(temp_2025_ma.index, temp_2025_ma, color="tab:red", linewidth=2,
        label="7-dagers glidende gjennomsnitt")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_title(f"Temperatur på {STATION_NAME} i 2025 med 7-dagers glidende gjennomsnitt")
ax.set_xlabel("Dato")
ax.set_ylabel("Temperatur (°C)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_xlim(temp_2025.index.min(), temp_2025.index.max())
ax.legend()

plt.tight_layout()
plt.show()

**Diskusjon:** _Hva gjør det glidende gjennomsnittet lettere å se enn de daglige verdiene?
Hva ville skjedd med kurven hvis vi brukte et lengre vindu, f.eks. 30 dager?_

## Kilder og bruk av KI

- Koden for å hente data bygger på eksempelet fra
  [frost.met.no](https://frost.met.no/python_example.html).
- Data: Meteorologisk institutt (MET), Frost API.
- _Beskriv her hvordan dere brukte KI-verktøy i arbeidet._